# Pharmaceutical Document RAG

A pharmaceutical document question-answering system built on `RAGPipeline`.

---

| Component | Details |
|---|---|
| **Embedding** | `sentence-transformers/all-MiniLM-L6-v2` |
| **Chunking** | Semantic (LlamaIndex) |
| **Retrieval** | Hybrid — Vector + BM25, reciprocal rerank |
| **LLM** | Mistral 7B Instruct (local, HuggingFace) |
| **OCR** | Tesseract fallback for scanned pages |
| **UI** | Gradio |

## 1. Install Dependencies

In [ ]:
%pip install -q pymupdf
%pip install -q llama-index llama-index-core
%pip install -q llama-index-embeddings-huggingface
%pip install -q llama-index-llms-llama-cpp
%pip install -q llama-index-retrievers-bm25
%pip install -q llama-cpp-python
%pip install -q sentence-transformers huggingface-hub torch
%pip install -q pytesseract pillow
%pip install -q "gradio>=6.9.0" --upgrade
%pip install -q "nest-asyncio>=1.6.0"

## 2. Imports

In [2]:
import sys
import os
from pathlib import Path

# Ensure the RAG directory is on the path so rag_pipeline can be imported
sys.path.insert(0, str(Path(".").resolve()))

from rag_pipeline import RAGPipeline
import gradio as gr

c:\Users\anjoe\OneDrive\Desktop\PfizerExtern\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


resource module not available on Windows


## 3. Initialize RAG Pipeline

Loads the GGUF model directly from disk — no download needed.

> **GPU offload:** `n_gpu_layers=-1` offloads all layers to GPU (CUDA). Set `n_gpu_layers=0` to run on CPU only.

In [3]:
MODEL_PATH = r"C:\LLM Models\Mistral\mistral-7b-instruct-v0.2.Q4_K_M.gguf"

rag = RAGPipeline(model_path=MODEL_PATH)
print("RAGPipeline initialized.")

RAGPipeline initialized.


## 4. Gradio UI

Upload a pharmaceutical PDF, click **Build Pipeline** to index it, then ask questions in the chat. Each answer includes source citations and per-chunk confidence scores.

In [ ]:
_pipeline_ready = False


def build_pipeline(pdf_file, classify_docs):
    """Index the uploaded PDF and mark the pipeline as ready."""
    global _pipeline_ready
    if pdf_file is None:
        return "No file uploaded."
    try:
        _pipeline_ready = False
        rag.build(pdf_file, classify_docs=classify_docs)
        _pipeline_ready = True
        label = os.path.basename(pdf_file)
        if classify_docs:
            label += " (doc types classified)"
        return f"Ready — {label}"
    except Exception as exc:
        return f"Error: {exc}"


def ask(question, history, classify):
    """Run a RAG query and return updated chat history + formatted sources."""
    if not question.strip():
        return history, "", "*Ask a question above.*"

    if not _pipeline_ready:
        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": "Please upload and build a document first."},
        ]
        return history, "", "*No sources — pipeline not ready.*"

    try:
        result = rag.query_with_sources(question, classify=classify)
    except Exception as exc:
        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": f"Error: {exc}"},
        ]
        return history, "", "*Error during retrieval.*"

    answer = result["answer"]
    sources = result["sources"]
    chunk_count = result["chunk_count"]
    query_category = result["query_category"]

    # Build sources panel
    sources_md = f"**{chunk_count} chunk(s) retrieved**"
    if query_category:
        sources_md += f" &nbsp;·&nbsp; Query classified as: **`{query_category}`**"
    sources_md += "\n\n---\n\n"

    for i, src in enumerate(sources, 1):
        confidence_str = f"{src['score']:.1f}%" if src["score"] is not None else "N/A"
        pharma_label = src.get("pharma_doc_type", "unknown")
        sources_md += (
            f"**Source {i}** &nbsp;·&nbsp; "
            f"`{src['file']}` &nbsp;·&nbsp; "
            f"Page **{src['page']}** &nbsp;·&nbsp; "
            f"Confidence: **{confidence_str}** &nbsp;·&nbsp; "
            f"Doc type: {src['doc_type']} &nbsp;·&nbsp; "
            f"Pharma type: **{pharma_label}**\n\n"
            f"> {src['text'][:300]}{'...' if len(src['text']) > 300 else ''}\n\n"
            f"---\n\n"
        )

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return history, "", sources_md


# ── Layout ────────────────────────────────────────────────────────────────────
with gr.Blocks(title="Pharma RAG") as demo:

    gr.Markdown("# Pharmaceutical Document RAG\nUpload a PDF, build the pipeline, then ask questions.")

    # Document ingestion row
    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], scale=3)
        with gr.Column(scale=2):
            classify_docs_toggle = gr.Checkbox(
                label="Classify document pages by pharma type",
                value=False,
                info="Enables targeted retrieval per document category. Adds one LLM call per page.",
            )
            build_btn = gr.Button("Build Pipeline", variant="primary", size="lg")
            status_box = gr.Textbox(
                label="Status",
                value="No document loaded.",
                interactive=False,
            )

    gr.Markdown("---")

    chatbot = gr.Chatbot(label="Chat", height=420)

    with gr.Row():
        question_input = gr.Textbox(
            label="Question",
            placeholder="e.g. What are the storage conditions?",
            scale=5,
        )
        ask_btn = gr.Button("Ask", variant="primary", scale=1)

    classify_query_toggle = gr.Checkbox(
        label="Classify query to restrict retrieval to matching document type",
        value=False,
        info="Uses the LLM to detect which pharma doc type best answers the query, then filters chunks accordingly.",
    )

    # Sources panel
    with gr.Accordion("Sources & Confidence", open=True):
        sources_display = gr.Markdown("*Sources will appear here after asking a question.*")

    # Event wiring
    build_btn.click(build_pipeline, inputs=[pdf_input, classify_docs_toggle], outputs=[status_box])
    ask_btn.click(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle],
        outputs=[chatbot, question_input, sources_display],
    )
    question_input.submit(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle],
        outputs=[chatbot, question_input, sources_display],
    )

demo.launch(share=False, theme=gr.themes.Soft())